# PadronizaÃ§Ã£o, auditoria de viÃ©s e avaliaÃ§Ã£o (UnivFD e FSD) no dataset polÃ­tico brasileiro
**PrÃ©-requisitos:** `real_oficial.zip` e a pasta `fake_geradas/` (do notebook 2) em `MyDrive/tcc_politicos/`.
**Funciona com ou sem GPU.** Sem GPU, o UnivFD roda em CPU (mais lento) e o FSD fica limitado a uma amostra (veja `FSD_N` na cÃ©lula 4).
**PadronizaÃ§Ã£o igual para as duas classes:** recorte central de 512Ã—512 com os pixels originais (sem reamostragem), sem metadados, **um Ãºnico JPEG q90**. Imagens com lado menor que 512 px sÃ£o descartadas e registradas.
**Auditoria:** antes de avaliar os detectores, testa se atalhos triviais (tamanho do arquivo, nitidez, cor) jÃ¡ separam real de falsa. Se separarem, o viÃ©s existe e precisa ser declarado.

In [ ]:
# 1) Setup
import os, io, subprocess, zipfile, glob, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv", shell=True, capture_output=True, text=True).stdout or "sem GPU (rodando em CPU)")
from google.colab import drive; drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/tcc_politicos"
def sh(c): r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:])
def unzip_fix(zpath, dest="/content"):   # zip criado no Windows usa "\" nos caminhos; converte para "/"
    with zipfile.ZipFile(zpath) as z:
        for i in z.infolist():
            n = i.filename.replace("\\", "/")
            if n.endswith("/"): continue
            p = os.path.join(dest, n); os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "wb") as f: f.write(z.read(i))
if not os.path.isdir("/content/real_oficial") or not os.listdir("/content/real_oficial"): unzip_fix(f"{ROOT}/real_oficial.zip")
sh("pip -q uninstall -y torchao 2>&1 | tail -1; pip -q install ftfy regex pillow-heif click tqdm huggingface_hub 2>&1 | tail -1")
sh("cd /content && rm -rf UniversalFakeDetect Forensic-Self-Descriptions-CVPR25 && git clone -q https://github.com/Yuheng-Li/UniversalFakeDetect && git clone -q https://github.com/ductai199x/Forensic-Self-Descriptions-CVPR25.git && git -C Forensic-Self-Descriptions-CVPR25 log -1 --format='FSD %H' && git -C UniversalFakeDetect log -1 --format='UnivFD %H'")
sh("pip -q install --no-deps -e /content/Forensic-Self-Descriptions-CVPR25 2>&1 | tail -1")

In [ ]:
# 2) Padroniza: crop central 512x512 nativo -> JPEG q90 unico. Reais e falsas pelo MESMO caminho.
STD = "/content/padronizado"; SOC = "/content/social"
for d in (f"{STD}/real", f"{STD}/fake", f"{SOC}/real", f"{SOC}/fake"): os.makedirs(d, exist_ok=True)
gl = pd.read_csv(f"{ROOT}/geradas_log.csv") if os.path.exists(f"{ROOT}/geradas_log.csv") else pd.DataFrame(columns=["arquivo", "gerador"])
gmap = dict(zip(gl.arquivo, gl.gerador))
def gerador_do_nome(fn):   # fallback se o log nao existir: o nome do arquivo comeca com o gerador (sdxl_1000.png)
    return gmap.get(fn, fn.split("_")[0])
def std512(im):
    im = im.convert("RGB"); w, h = im.size
    if min(w, h) < 512: return None
    l, t = (w - 512) // 2, (h - 512) // 2
    return im.crop((l, t, l + 512, t + 512))
items = [(f, 0, "commons", "real") for f in sorted(glob.glob("/content/real_oficial/*.jpg"))] + \
        [(f, 1, "gerada", gerador_do_nome(os.path.basename(f))) for f in sorted(glob.glob(f"{ROOT}/fake_geradas/*.png"))]
meta, descartadas = [], []
for f, y, src, gen in tqdm(items):
    im = std512(Image.open(f))
    if im is None: descartadas.append(f); continue
    name = ("fake" if y else "real"); base = os.path.splitext(os.path.basename(f))[0]
    p1 = f"{STD}/{name}/{base}.jpg"; im.save(p1, "JPEG", quality=90)                   # condicao A: padrao
    p2 = f"{SOC}/{name}/{base}.jpg"; im.resize((384, 384), Image.LANCZOS).save(p2, "JPEG", quality=75)  # condicao B: tipo rede social
    pol = "_".join(base.split("_")[1:]) if y == 0 else ""
    meta.append(dict(base=base, label=y, origem=src, gerador=gen, politico=pol, std=p1, soc=p2, kb=os.path.getsize(p1) / 1024))
M = pd.DataFrame(meta); M.to_csv(f"{ROOT}/padronizado_meta.csv", index=False)
print(len(M), "imagens;", len(descartadas), "descartadas (lado menor < 512)"); print(M.groupby(["label", "gerador"]).size())

In [ ]:
# 3) AUDITORIA DE VIES: atalhos triviais separam real de falsa?  (AUC ~0,5 = bom; > 0,7 = ainda ha vies). Roda em CPU.
from scipy import ndimage
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def feats(p):
    a = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255; g = a.mean(2)
    lap = ndimage.laplace(g); gx, gy = np.gradient(g)
    return [os.path.getsize(p) / 1024, lap.var(), np.hypot(gx, gy).mean(), *a.mean((0, 1)), *a.std((0, 1)),
            (a.max(2) - a.min(2)).mean(), np.abs(np.fft.fftshift(np.fft.fft2(g)))[240:272, 240:272].mean()]
names = ["tam_kb", "laplaciano", "gradiente", "R", "G", "B", "sdR", "sdG", "sdB", "saturacao", "baixa_freq"]
X = np.array([feats(p) for p in tqdm(M["std"])]); y = M.label.values
cv = StratifiedKFold(5, shuffle=True, random_state=0)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pr = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
print("AUC (todas as features juntas, 5-fold):", round(roc_auc_score(y, pr), 3))
print(pd.Series({n: round(max(roc_auc_score(y, X[:, i]), 1 - roc_auc_score(y, X[:, i])), 3) for i, n in enumerate(names)}).sort_values(ascending=False).to_string())
print("-> Se algum valor for > 0,7, o conjunto ainda tem atalho e isso DEVE ser declarado nas limitacoes.")
pd.DataFrame(X, columns=names).assign(label=y, gerador=M.gerador.values).to_csv(f"{ROOT}/auditoria_features.csv", index=False)

In [ ]:
# 4) Detectores. Com GPU roda tudo. SEM GPU: UnivFD em CPU (mais lento); FSD so numa amostra (FSD_N) ou desligado (RUN_FSD=False).
import sys, torch, torchvision.transforms as T
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("dispositivo:", DEVICE)
RUN_FSD = True                       # False = so UnivFD
FSD_N = None if DEVICE == "cuda" else 120   # em CPU: FSD em 120 imagens (60 reais + 60 falsas). Use None para todas (so com GPU)
CONDS = [("A", "std", 512), ("B", "soc", 384)] if DEVICE == "cuda" else [("A", "std", 512)]   # em CPU so a condicao A
sys.path.insert(0, "/content/UniversalFakeDetect"); os.chdir("/content/UniversalFakeDetect")
from models import get_model
um = get_model("CLIP:ViT-L/14"); um.fc.load_state_dict(torch.load("pretrained_weights/fc_weights.pth", map_location="cpu")); um.eval().to(DEVICE)
tf = T.Compose([T.CenterCrop(224), T.ToTensor(), T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
@torch.no_grad()
def u_score(p): return um(tf(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)).sigmoid().item()
for col, path_col, ms in CONDS:
    if f"u_{col}" in M and M[f"u_{col}"].notna().all(): continue
    M[f"u_{col}"] = [u_score(p) for p in tqdm(M[path_col], desc=f"UnivFD {col}")]
    M.to_csv(f"{ROOT}/resultados_politico.csv", index=False)
if RUN_FSD:
    sys.path.insert(0, "/content/Forensic-Self-Descriptions-CVPR25")
    from fsd import FSDDetector
    det = FSDDetector.load(device=DEVICE)
    def f_score(p, ms):
        det.config["fsd"]["max_size"] = ms
        try: z = float(det.score(p).z_score)
        except Exception as e: z = np.nan
        if DEVICE == "cuda": torch.cuda.empty_cache()
        return z
    if FSD_N:
        rng = np.random.default_rng(0)
        idx = np.concatenate([rng.choice(M.index[M.label == c], FSD_N // 2, replace=False) for c in (0, 1)])
    else: idx = M.index.values
    for col, path_col, ms in CONDS:
        if f"f_{col}" not in M: M[f"f_{col}"] = np.nan
        for i in tqdm(idx, desc=f"FSD {col}"):
            if np.isnan(M.at[i, f"f_{col}"]): M.at[i, f"f_{col}"] = f_score(M.at[i, path_col], ms)
        M.to_csv(f"{ROOT}/resultados_politico.csv", index=False)
print("pronto")

In [ ]:
# 5) Metricas com IC95% (bootstrap), por gerador e com limiar escolhido em metade e testado na outra (sem otimismo)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, average_precision_score, confusion_matrix
def boot(y, s, n=2000, seed=0):
    rng = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(roc_auc_score(y[i], s[i]))
    return np.percentile(v, [2.5, 97.5])
y = M.label.values; R = []
for det_name, col, sg, thr in [("UnivFD", "u", 1, 0.5), ("FSD", "f", -1, -2.0)]:
    for cond in ("A", "B"):
        if f"{col}_{cond}" not in M or M[f"{col}_{cond}"].notna().sum() < 20: continue
        s_raw = M[f"{col}_{cond}"].values; ok = ~np.isnan(s_raw); yy = y[ok]; s = sg * s_raw[ok]
        lo, hi = boot(yy, s); pred = (s_raw[ok] > thr) if col == "u" else (s_raw[ok] < thr)
        tn, fp, fn, tp = confusion_matrix(yy, pred).ravel()
        rng = np.random.default_rng(1); idx = rng.permutation(len(yy)); a, b = idx[:len(idx)//2], idx[len(idx)//2:]
        fpr, tpr, th = roc_curve(yy[a], s[a]); t = th[np.argmax(tpr - fpr)]; acc_held = ((s[b] >= t) == yy[b]).mean()
        R.append(dict(detector=det_name, condicao=cond, n=int(ok.sum()), AUC=roc_auc_score(yy, s), IC95=f"[{lo:.2f}, {hi:.2f}]", AP=average_precision_score(yy, s),
                      acc_limiar_padrao=(tp+tn)/len(yy), recall_fake=tp/(tp+fn), especificidade=tn/(tn+fp), acc_limiar_ajustado_heldout=acc_held))
R = pd.DataFrame(R).round(3); print(R.to_string(index=False)); R.to_csv(f"{ROOT}/resumo_politico.csv", index=False)
rows = []
for gname in sorted(set(M.gerador[M.label == 1])):
    m = M[(M.gerador == gname) | (M.label == 0)]; r = dict(gerador=gname, n_fake=int((M.gerador == gname).sum()), AUC_UnivFD_A=roc_auc_score(m.label, m.u_A))
    if "f_A" in M:
        mm = m.dropna(subset=["f_A"])
        if mm.label.nunique() == 2 and len(mm) >= 20: r["AUC_FSD_A"] = roc_auc_score(mm.label, -mm.f_A)
    rows.append(r)
G = pd.DataFrame(rows).round(3); print(G.to_string(index=False)); G.to_csv(f"{ROOT}/auc_por_gerador_politico.csv", index=False)
conds = [c for c in ("A", "B") if f"u_{c}" in M]
fig, ax = plt.subplots(1, len(conds), figsize=(5.5 * len(conds), 4.2), squeeze=False)
for k, cond in enumerate(conds):
    a_ = ax[0][k]
    for nm, col, sg in (("UnivFD", "u", 1), ("FSD", "f", -1)):
        if f"{col}_{cond}" not in M: continue
        d = M.dropna(subset=[f"{col}_{cond}"])
        if d.label.nunique() < 2: continue
        s = sg * d[f"{col}_{cond}"].values; fpr, tpr, _ = roc_curve(d.label, s); a_.plot(fpr, tpr, label=f"{nm} (AUC={roc_auc_score(d.label, s):.2f}, n={len(d)})")
    a_.plot([0, 1], [0, 1], "k--"); a_.set(title=f"ROC, condicao {cond}", xlabel="Falso positivo", ylabel="Recall (fake)"); a_.legend()
plt.tight_layout(); plt.savefig(f"{ROOT}/roc_politico.png", dpi=200); plt.show()